[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentOpt/Trace-Bench/blob/main/notebooks/08_multiobjective_gsm8k.ipynb)

# Trace-Bench — Multi-Objective GSM8K

This notebook benchmarks three trainer algorithms on the **GSM8K** multi-objective
task, comparing **weighted** vs **Pareto** selection modes across two LLM backends.

**Task**: optimize a system prompt for solving grade-school math word problems.
The trainable parameter is the system prompt inside a `BasicLearner` agent;
the LLM generates step-by-step solutions ending with `#### <answer>`.

| Objective | Weight | Description | Direction |
|-----------|--------|-------------|-----------|
| `error` | 1.0 | 0.0 correct, 1.0 wrong (exact-match on final answer) | minimize |
| `tokens_in` | 1e-3 | Prompt tokens (from provider metadata or whitespace estimate) | minimize |
| `tokens_out` | 1e-3 | Completion tokens | minimize |

**Architecture** (from Xavier's design):
- `UsageTrackingLLM` — wraps the base LLM, intercepts `__call__()` to record
  `prompt_tokens` / `completion_tokens` via `contextvars.ContextVar` (thread-safe).
- `GSM8KGuide` — scores by exact match on the extracted final numeric answer.
- `TokenUsageAugmentingGuide` — wraps `GSM8KGuide` + `UsageTrackingLLM`,
  merges `tokens_in` and `tokens_out` into the score dict.

**Trainers**: BasicSearchAlgorithm, BeamsearchAlgorithm, PrioritySearch — all with `batch_size=1`.

**Models**: `grok-4.1-fast` and `deepseek` (v3.2 or chat).
Supports both **OpenRouter** (`OPENROUTER_API_KEY`) and **direct provider keys**
(`XAI_API_KEY` + `DEEPSEEK_API_KEY`) via LiteLLM. Falls back to stub/DummyLLM
if no keys are found. Place keys in a `.env` file or export them in your shell.

## Expected Outputs

- Up to 12 completed jobs (3 trainers × 2 modes × 2 models) in two separate runs.
- Results table comparing all 12 runs grouped by model.
- Per-metric progression plots: error, tokens_in, tokens_out.
- Scatter plot of error vs tokens_out (primary trade-off), faceted by model.
- Parallel coordinates plot showing all 3 objectives simultaneously.
- Model comparison: side-by-side bar chart + token efficiency analysis.

In [ ]:
# Setup: persistent output dir, API key detection, model config
from datetime import date
from pathlib import Path
import os, sys

# --- Load .env (local dev) ---------------------------------------------------
try:
    from dotenv import load_dotenv
    # Walk up to find .env (repo root or parent)
    _here = Path.cwd()
    for _candidate in [_here, _here.parent, _here.parent.parent]:
        _envfile = _candidate / ".env"
        if _envfile.is_file():
            load_dotenv(_envfile, override=False)
            print(f"Loaded .env from {_envfile}")
            break
except ImportError:
    pass  # dotenv not installed — rely on shell environment

# --- Detect environment: Colab vs local --------------------------------------
ON_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except Exception:
    pass

if ON_COLAB:
    REPO_ROOT = Path("/content/Trace-Bench")
    OPENTRACE_ROOT = Path("/content/OpenTrace")
else:
    REPO_ROOT = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd().parent
    if not (REPO_ROOT / "trace_bench").is_dir():
        REPO_ROOT = Path.cwd()
    OPENTRACE_ROOT = REPO_ROOT.parent / "OpenTrace"


def bench_dir(project="bench", sub="trace_bench_gsm8k"):
    if ON_COLAB:
        drive_root = Path("/content/drive/MyDrive")
        root = drive_root if drive_root.is_dir() else Path("/content/bench")
    else:
        root = REPO_ROOT / "runs"
    out = root / project / date.today().isoformat() / sub
    out.mkdir(parents=True, exist_ok=True)
    return str(out)


RUNS_DIR = bench_dir()
os.environ["RUNS_DIR"] = RUNS_DIR
print("Runs dir:", RUNS_DIR)
print(f"Repo root: {REPO_ROOT}")
print(f"OpenTrace: {OPENTRACE_ROOT}")

# --- Auto-detect API provider ------------------------------------------------
# Priority: OpenRouter > Colab secrets > Direct provider keys > stub
#
# 1. OpenRouter  — models prefixed "openrouter/..."
# 2. Direct      — XAI_API_KEY + DEEPSEEK_API_KEY -> native LiteLLM model IDs
# 3. Stub        — DummyLLM fallback

PROVIDER = None
MODELS = []

# Try OpenRouter first (Xavier's default setup)
_or_key = os.environ.get("OPENROUTER_API_KEY", "")
if not _or_key:
    try:
        from google.colab import userdata
        _or_key = userdata.get("OPENROUTER_API_KEY") or ""
    except Exception:
        pass

if _or_key:
    PROVIDER = "openrouter"
    os.environ["OPENROUTER_API_KEY"] = _or_key
    os.environ["OPENAI_API_KEY"] = _or_key
    os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"
    os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
    _model_1 = os.environ.get("TRACE_LITELLM_MODEL", "")
    _model_2 = os.environ.get("TRACE_LITELLM_MODEL_2", "")
    if not _model_1:
        print("Set TRACE_LITELLM_MODEL env var for real-mode runs. Using stub mode.")
    if not _model_2:
        print("Set TRACE_LITELLM_MODEL_2 env var for second model comparison.")
    MODELS = [m for m in [_model_1, _model_2] if m]

# Try direct provider keys (XAI + DeepSeek via native LiteLLM routing)
if not PROVIDER:
    # Handle common XIA_API_KEY typo
    if not os.environ.get("XAI_API_KEY") and os.environ.get("XIA_API_KEY"):
        os.environ["XAI_API_KEY"] = os.environ["XIA_API_KEY"]

    _xai_key = os.environ.get("XAI_API_KEY", "")
    _ds_key = os.environ.get("DEEPSEEK_API_KEY", "")

    if _xai_key or _ds_key:
        PROVIDER = "direct"
        MODELS = []
        if _xai_key:
            _model_1 = os.environ.get("TRACE_LITELLM_MODEL", "")
            if _model_1:
                MODELS.append(_model_1)
        if _ds_key:
            _model_2 = os.environ.get("TRACE_LITELLM_MODEL_2", "")
            if _model_2:
                MODELS.append(_model_2)
        if not MODELS:
            print("Set TRACE_LITELLM_MODEL env var for real-mode runs. Using stub mode.")

# Fallback to stub
if not PROVIDER or not MODELS:
    PROVIDER = "stub"
    MODELS = []

# Set mode and backend
if PROVIDER in ("openrouter", "direct"):
    os.environ["TRACE_DEFAULT_LLM_BACKEND"] = "LiteLLM"
    MODE = "real"
    print(f"Provider: {PROVIDER} — running in REAL mode")
    print(f"Models: {MODELS}")
else:
    MODE = "stub"
    print("WARNING: No API keys found. Falling back to STUB mode.")
    print("         Stub results use DummyLLM — not real optimization.")

os.environ["TB_MODE"] = MODE
print(f"\nMode: {MODE}")

In [ ]:
# Clone repos + install deps (Colab only — skipped locally)
if ON_COLAB:
    !git clone --depth 1 https://github.com/AgentOpt/Trace-Bench.git /content/Trace-Bench 2>/dev/null || (cd /content/Trace-Bench && git pull --ff-only)
    !git clone --depth 1 --branch experimental https://github.com/AgentOpt/OpenTrace.git /content/OpenTrace 2>/dev/null || (cd /content/OpenTrace && git pull)

    # Add repos to sys.path so trace_bench and opto are importable
    sys.path.insert(0, str(REPO_ROOT))
    sys.path.insert(0, str(OPENTRACE_ROOT))

    %cd /content/Trace-Bench
    !python -m pip install -q pyyaml pytest numpy matplotlib pandas litellm==1.75.0 datasets

    # Verify task module is available after clone
    _task_mod = REPO_ROOT / "trace_bench" / "examples" / "multiobjective_gsm8k.py"
    assert _task_mod.exists(), f"ERROR: {_task_mod} not found — ensure main branch is up to date."
    print(f"Task module verified: {_task_mod}")
else:
    os.chdir(str(REPO_ROOT))
    print(f"Working directory: {Path.cwd()}")
    print("Skipping clone/install — running locally.")

## 1. Load Task Bundle

Verify the GSM8K multi-objective task loads correctly.
The bundle uses `BasicLearner` (trainable system prompt → LLM → answer)
with `UsageTrackingLLM` for token counting.

In [ ]:
import sys

sys.path.insert(0, str(OPENTRACE_ROOT))
sys.path.insert(0, str(REPO_ROOT))

from trace_bench.registry import load_task_bundle

bundle = load_task_bundle(
    "internal:multiobjective_gsm8k",
    "benchmarks/LLM4AD/benchmark_tasks",
    eval_kwargs={"objective_mode": "weighted"},
)

print("Bundle keys:", sorted(bundle.keys()))
print(f"Param type: {type(bundle['param']).__name__}")
print(f"Guide type: {type(bundle['guide']).__name__}")
print(f"Metadata: {bundle['metadata']}")
print(f"Train examples: {len(bundle['train_dataset']['inputs'])}")

oc = bundle["objective_config"]
print(f"\nObjectiveConfig:")
print(f"  mode     = {oc.mode}")
print(f"  weights  = {dict(oc.weights)}")
print(f"  minimize = {set(oc.minimize)}")

# Show first training example
q0 = bundle["train_dataset"]["inputs"][0]
a0 = bundle["train_dataset"]["infos"][0]
print(f"\nExample question: {q0[:120]}...")
print(f"Example answer (last line): ...{a0.strip().splitlines()[-1]}")

# Show the trainable system prompt
params = list(bundle["param"].parameters())
print(f"\nTrainable parameters: {len(params)}")
for p in params:
    print(f"  {p.description}: {str(p.data)[:100]}...")

del bundle

## 2. Run Experiments

3 trainers × 2 modes (weighted / pareto) × 2 models = **12 experiments**.

Each model is run as a separate `trace_bench run` invocation with the
corresponding `TRACE_LITELLM_MODEL` environment variable.

Training params are conservative (`num_epochs=2`, `batch_size=1`) since
each step involves LLM API calls for both the optimizer and the GSM8K agent.

In [ ]:
# Write the YAML config (shared by both model runs)
import pathlib

yaml_content = """\
mode: {mode}
seeds: [42]
max_workers: 6  # 6 parallel jobs — bottleneck is API latency, not CPU
resume: auto
job_timeout: 1200

tasks:
  - id: "internal:multiobjective_gsm8k"
    eval_kwargs:
      objective_mode: "weighted"

  - id: "internal:multiobjective_gsm8k"
    eval_kwargs:
      objective_mode: "pareto"

trainers:
  - id: BasicSearchAlgorithm
    params_variants:
      - num_proposals: 4
        num_epochs: 2
        batch_size: 1

  - id: BeamsearchAlgorithm
    params_variants:
      - beam_width: 2
        num_proposals: 4
        max_depth: 2
        batch_size: 1

  - id: PrioritySearch
    params_variants:
      - num_candidates: 4
        num_proposals: 2
        batch_size: 1
""".format(mode=MODE)

config_path = pathlib.Path(RUNS_DIR) / "gsm8k.yaml"
config_path.write_text(yaml_content)
print(f"Config written to {config_path}")
print(f"Mode: {MODE}")
print(yaml_content)

In [ ]:
# Run Model 1 (first model in MODELS list)
import subprocess, os, sys

if not MODELS:
    print("No models configured — skipping. Running in stub mode.")
    MODEL_1 = "stub"
    MODEL_1_TAG = "stub"
else:
    MODEL_1 = MODELS[0]
    MODEL_1_TAG = MODEL_1.split("/")[-1]  # e.g. "grok-4.1-fast"

print(f"=== Running GSM8K with {MODEL_1} (provider: {PROVIDER}) ===")

env = dict(os.environ)
env["PYTHONPATH"] = str(OPENTRACE_ROOT) + ":" + env.get("PYTHONPATH", "")
if MODEL_1 != "stub":
    env["TRACE_LITELLM_MODEL"] = MODEL_1

result = subprocess.run(
    [
        sys.executable, "-m", "trace_bench", "run",
        "--config", str(config_path),
        "--runs-dir", f"{RUNS_DIR}/{MODEL_1_TAG}",
    ],
    cwd=str(REPO_ROOT),
    env=env,
    capture_output=True,
    text=True,
    timeout=3600,
)

output = result.stdout
print(output[-3000:] if len(output) > 3000 else output)
if result.returncode != 0:
    print(f"\nSTDERR (last 2000 chars):\n{result.stderr[-2000:]}")
print(f"\nReturn code: {result.returncode}")

In [ ]:
# Run Model 2 (second model in MODELS list, if available)
if len(MODELS) < 2:
    print("Only one model configured — skipping Model 2 run.")
    MODEL_2 = None
    MODEL_2_TAG = None
else:
    MODEL_2 = MODELS[1]
    MODEL_2_TAG = MODEL_2.split("/")[-1]  # e.g. "deepseek-v3.2" or "deepseek-chat"
    print(f"=== Running GSM8K with {MODEL_2} (provider: {PROVIDER}) ===")

    env = dict(os.environ)
    env["PYTHONPATH"] = str(OPENTRACE_ROOT) + ":" + env.get("PYTHONPATH", "")
    env["TRACE_LITELLM_MODEL"] = MODEL_2

    result = subprocess.run(
        [
            sys.executable, "-m", "trace_bench", "run",
            "--config", str(config_path),
            "--runs-dir", f"{RUNS_DIR}/{MODEL_2_TAG}",
        ],
        cwd=str(REPO_ROOT),
        env=env,
        capture_output=True,
        text=True,
        timeout=3600,
    )

    output = result.stdout
    print(output[-3000:] if len(output) > 3000 else output)
    if result.returncode != 0:
        print(f"\nSTDERR (last 2000 chars):\n{result.stderr[-2000:]}")
    print(f"\nReturn code: {result.returncode}")

## 3. Results Table

Load `results.csv` from both model runs and display a unified comparison.

In [ ]:
import pathlib, json, pandas as pd, numpy as np


def latest_run(root):
    """Find the most recent run directory under root."""
    root = pathlib.Path(root)
    if not root.exists():
        return None
    candidates = sorted(
        [p for p in root.iterdir() if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
    )
    for p in reversed(candidates):
        if (p / "results.csv").exists():
            return p
    return None


def load_run(root, model_tag):
    """Load results.csv from a model run, adding model column."""
    run_dir = latest_run(root)
    if run_dir is None:
        print(f"WARNING: No run found for {model_tag} in {root}")
        return None, None
    df = pd.read_csv(run_dir / "results.csv")
    df["model"] = model_tag
    df["objective_mode"] = df["eval_kwargs"].apply(
        lambda x: json.loads(x).get("objective_mode", "?") if isinstance(x, str) else "?"
    )
    return df, run_dir


# Load both runs
frames = []
run_dirs = {}

for model_name in MODELS:
    tag = model_name.split("/")[-1]
    df_model, rd = load_run(f"{RUNS_DIR}/{tag}", tag)
    if df_model is not None:
        frames.append(df_model)
        run_dirs[tag] = rd
        print(f"{tag}: {len(df_model)} jobs, status={df_model['status'].value_counts().to_dict()}")

if not frames:
    print("No results found. Check that the run cells completed.")
    df_all = pd.DataFrame()
else:
    df_all = pd.concat(frames, ignore_index=True)
    print(f"\nTotal jobs: {len(df_all)}")

if not df_all.empty:
    display_cols = [
        "model", "trainer_id", "objective_mode", "status",
        "score_initial", "score_final", "score_best", "time_seconds",
    ]
    df_all[display_cols]

## 4. Per-Metric Evaluation

Parse TensorBoard event files for the three GSM8K objectives:
`error`, `tokens_in`, `tokens_out`.

In [ ]:
import struct


def read_tb_scalars(logdir):
    """Read scalar summaries from TensorBoard event files."""
    scalars = []
    logdir = pathlib.Path(logdir)
    if not logdir.exists():
        return scalars
    for event_file in sorted(logdir.glob("events.out.tfevents.*")):
        try:
            data = event_file.read_bytes()
            offset = 0
            while offset < len(data):
                if offset + 12 > len(data):
                    break
                length = struct.unpack("Q", data[offset:offset + 8])[0]
                offset += 12
                if offset + length + 4 > len(data):
                    break
                record = data[offset:offset + length]
                offset += length + 4
                try:
                    from tensorflow.core.util import event_pb2
                    event = event_pb2.Event()
                    event.ParseFromString(record)
                    if event.HasField("summary"):
                        for v in event.summary.value:
                            scalars.append((v.tag, event.step, v.simple_value))
                except ImportError:
                    pass
        except Exception:
            continue
    return scalars


def collect_tb_metrics_gsm8k(run_dir, df_sub):
    """Extract error/tokens_in/tokens_out histories from TB logs."""
    records = []
    for _, row in df_sub.iterrows():
        job_id = row["job_id"]
        tb_rel = row.get("tb_logdir", f"jobs/{job_id}/tb")
        tb_dir = pathlib.Path(run_dir) / tb_rel
        scalars = read_tb_scalars(tb_dir)

        error_h = [(s, v) for tag, s, v in scalars if tag.endswith("/error") or tag == "error"]
        tin_h = [(s, v) for tag, s, v in scalars if "tokens_in" in tag]
        tout_h = [(s, v) for tag, s, v in scalars if "tokens_out" in tag]

        records.append({
            "job_id": job_id,
            "trainer_id": row["trainer_id"],
            "objective_mode": row["objective_mode"],
            "model": row["model"],
            "error_history": error_h,
            "tokens_in_history": tin_h,
            "tokens_out_history": tout_h,
            "error_final": error_h[-1][1] if error_h else None,
            "tokens_in_final": tin_h[-1][1] if tin_h else None,
            "tokens_out_final": tout_h[-1][1] if tout_h else None,
        })
    return records


# Collect TB metrics for all runs
all_tb_metrics = []
for tag, rd in run_dirs.items():
    df_sub = df_all[df_all["model"] == tag]
    metrics = collect_tb_metrics_gsm8k(rd, df_sub)
    all_tb_metrics.extend(metrics)

has_tb = any(m["error_final"] is not None for m in all_tb_metrics)
print(f"TensorBoard metrics available: {has_tb}")

## 5. Score Progression

Plot the scalar validation score over training steps, grouped by model.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 11})

colors = {
    "BasicSearchAlgorithm": "#2196F3",
    "BeamsearchAlgorithm": "#FF9800",
    "PrioritySearch": "#4CAF50",
}

if df_all.empty:
    print("No data to plot.")
elif has_tb:
    model_tags = sorted(run_dirs.keys())
    n_models = len(model_tags)
    fig, axes = plt.subplots(n_models, 2, figsize=(14, 5 * n_models), squeeze=False)

    for row_idx, model_tag in enumerate(model_tags):
        for col_idx, mode_name in enumerate(["weighted", "pareto"]):
            ax = axes[row_idx, col_idx]
            for m in all_tb_metrics:
                if m["model"] != model_tag or m["objective_mode"] != mode_name:
                    continue
                rd = run_dirs[model_tag]
                tb_rel = f"jobs/{m['job_id']}/tb"
                scalars = read_tb_scalars(pathlib.Path(rd) / tb_rel)
                val = [(s, v) for tag, s, v in scalars
                       if "validation" in tag.lower() and "score/" not in tag.lower()]
                if not val:
                    val = [(s, v) for tag, s, v in scalars if "objective" in tag.lower()]
                if val:
                    steps, scores = zip(*val)
                    ax.plot(steps, scores, marker="o", ms=4,
                            label=m["trainer_id"],
                            color=colors.get(m["trainer_id"], "gray"))

            ax.set_title(f"{model_tag} — {mode_name}")
            ax.set_xlabel("Training Step")
            ax.set_ylabel("Validation Score")
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)

    fig.suptitle("GSM8K — Score Progression", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No TensorBoard data — showing bar chart of best scores.")
    model_tags = sorted(df_all["model"].unique())
    fig, axes = plt.subplots(1, len(model_tags), figsize=(7 * len(model_tags), 5),
                             squeeze=False, sharey=True)

    for col_idx, model_tag in enumerate(model_tags):
        ax = axes[0, col_idx]
        sub = df_all[df_all["model"] == model_tag]
        trainers = sub["trainer_id"].unique()
        modes = ["weighted", "pareto"]
        x = np.arange(len(trainers))
        width = 0.35

        for i, mode_name in enumerate(modes):
            mode_sub = sub[sub["objective_mode"] == mode_name]
            vals = [
                mode_sub[mode_sub["trainer_id"] == t]["score_best"].values[0]
                if len(mode_sub[mode_sub["trainer_id"] == t]) > 0 else 0
                for t in trainers
            ]
            ax.bar(x + i * width, vals, width, label=mode_name)

        ax.set_title(model_tag)
        ax.set_xlabel("Trainer")
        ax.set_ylabel("Score (best)")
        ax.set_xticks(x + width / 2)
        short = [t.replace("Algorithm", "") for t in trainers]
        ax.set_xticklabels(short, fontsize=9)
        ax.legend()
        ax.grid(True, alpha=0.3, axis="y")

    fig.suptitle("GSM8K — Best Score by Trainer & Mode", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 6. Per-Metric Progression (error + tokens_in + tokens_out)

Plot all three objective trajectories from TensorBoard logs,
or show summary bar charts if TB is unavailable.

In [ ]:
if df_all.empty:
    print("No data to plot.")
elif has_tb:
    model_tags = sorted(run_dirs.keys())
    n_models = len(model_tags)
    # 3 metrics x n_models rows, 2 mode columns
    fig, axes = plt.subplots(3 * n_models, 2, figsize=(14, 4 * 3 * n_models),
                             squeeze=False)

    metric_keys = [
        ("error_history", "error (lower = better)"),
        ("tokens_in_history", "tokens_in (lower = better)"),
        ("tokens_out_history", "tokens_out (lower = better)"),
    ]

    for m_idx, model_tag in enumerate(model_tags):
        for col, mode_name in enumerate(["weighted", "pareto"]):
            for met_idx, (met_key, met_label) in enumerate(metric_keys):
                ax = axes[m_idx * 3 + met_idx, col]

                for m in all_tb_metrics:
                    if m["model"] != model_tag or m["objective_mode"] != mode_name:
                        continue
                    history = m[met_key]
                    if history:
                        steps, vals = zip(*history)
                        ax.plot(steps, vals, marker="o", ms=3,
                                label=m["trainer_id"],
                                color=colors.get(m["trainer_id"], "gray"))

                ax.set_title(f"{model_tag} — {met_label} ({mode_name})")
                ax.set_xlabel("Step")
                ax.set_ylabel(met_label.split(" (")[0])
                ax.legend(fontsize=7)
                ax.grid(True, alpha=0.3)

    fig.suptitle("GSM8K — Per-Metric Progression (3 objectives)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No TensorBoard data — per-metric progression not available.")
    print("Run with TensorFlow installed for detailed metric tracking.")

## 7. Scatter Plot: error vs tokens_out

The primary trade-off: lower error (better accuracy) vs fewer output tokens
(cheaper/faster). Color by trainer, marker by mode, faceted by model.
Ideal: bottom-left (low error, few tokens).

In [ ]:
if df_all.empty:
    print("No data to plot.")
elif not has_tb:
    print("Scatter plot requires TensorBoard metrics (per-objective final values).")
    print("Showing score_best comparison instead.")
    print(df_all[["model", "trainer_id", "objective_mode", "score_best", "time_seconds"]])
else:
    model_tags = sorted(run_dirs.keys())
    n_models = len(model_tags)
    fig, axes = plt.subplots(1, max(n_models, 1), figsize=(7 * max(n_models, 1), 6),
                             squeeze=False)

    markers = {"weighted": "o", "pareto": "s"}

    for col_idx, model_tag in enumerate(model_tags):
        ax = axes[0, col_idx]
        seen_labels = set()

        for m in all_tb_metrics:
            if m["model"] != model_tag:
                continue
            err = m["error_final"]
            tout = m["tokens_out_final"]
            if err is None or tout is None:
                continue

            trainer_short = m["trainer_id"].replace("Algorithm", "")
            full_label = f"{trainer_short} ({m['objective_mode']})"

            ax.scatter(
                err, tout,
                c=colors.get(m["trainer_id"], "gray"),
                marker=markers.get(m["objective_mode"], "o"),
                s=120, edgecolors="black", linewidths=0.5,
                label=full_label if full_label not in seen_labels else None,
                zorder=5,
            )
            seen_labels.add(full_label)

        ax.set_xlabel("error (lower = better)")
        ax.set_ylabel("tokens_out (lower = better)")
        ax.set_title(model_tag)
        ax.legend(fontsize=8, loc="best")
        ax.grid(True, alpha=0.3)

    fig.suptitle("GSM8K — error vs tokens_out (Final)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 8. Parallel Coordinates: All 3 Objectives

Visualize all three objectives simultaneously using parallel coordinates.
Each line represents one experiment; axes are normalized to [0, 1].

In [ ]:
if df_all.empty or not has_tb:
    print("Parallel coordinates requires TensorBoard metrics.")
else:
    # Build data for parallel coordinates
    pc_rows = []
    for m in all_tb_metrics:
        if m["error_final"] is None:
            continue
        pc_rows.append({
            "label": f"{m['model']} / {m['trainer_id'].replace('Algorithm', '')} / {m['objective_mode']}",
            "trainer": m["trainer_id"],
            "model": m["model"],
            "mode": m["objective_mode"],
            "error": m["error_final"],
            "tokens_in": m["tokens_in_final"] or 0,
            "tokens_out": m["tokens_out_final"] or 0,
        })

    if not pc_rows:
        print("No complete metric data for parallel coordinates.")
    else:
        df_pc = pd.DataFrame(pc_rows)
        obj_cols = ["error", "tokens_in", "tokens_out"]

        # Normalize each objective to [0, 1]
        df_norm = df_pc.copy()
        for col in obj_cols:
            lo, hi = df_pc[col].min(), df_pc[col].max()
            if hi > lo:
                df_norm[col] = (df_pc[col] - lo) / (hi - lo)
            else:
                df_norm[col] = 0.5

        fig, ax = plt.subplots(figsize=(10, 6))
        x_pos = np.arange(len(obj_cols))

        for _, row in df_norm.iterrows():
            y_vals = [row[c] for c in obj_cols]
            c = colors.get(row["trainer"], "gray")
            ls = "-" if row["mode"] == "weighted" else "--"
            alpha = 0.9 if "grok" in row["model"] else 0.5
            ax.plot(x_pos, y_vals, marker="o", ms=6, color=c,
                    linestyle=ls, alpha=alpha, linewidth=2)

        ax.set_xticks(x_pos)
        ax.set_xticklabels([f"{c}\n(lower = better)" for c in obj_cols])
        ax.set_ylabel("Normalized value [0=best, 1=worst]")
        ax.set_title("GSM8K — Parallel Coordinates (all 3 objectives)")
        ax.grid(True, alpha=0.3, axis="y")

        # Custom legend
        from matplotlib.lines import Line2D
        legend_elements = []
        for t, c in colors.items():
            legend_elements.append(Line2D([0], [0], color=c, lw=2,
                                          label=t.replace("Algorithm", "")))
        legend_elements.append(Line2D([0], [0], color="gray", ls="-", lw=2,
                                      label="weighted"))
        legend_elements.append(Line2D([0], [0], color="gray", ls="--", lw=2,
                                      label="pareto"))
        ax.legend(handles=legend_elements, fontsize=8, loc="upper right")

        plt.tight_layout()
        plt.show()

        print("\nRaw objective values:")
        df_pc[["label", "error", "tokens_in", "tokens_out"]].sort_values("error")

## 9. Model Comparison

Side-by-side bar chart of `score_best` per model + token efficiency analysis.

In [ ]:
if df_all.empty:
    print("No data to compare.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    model_tags = sorted(df_all["model"].unique())

    for ax, mode_name in zip(axes, ["weighted", "pareto"]):
        mode_df = df_all[df_all["objective_mode"] == mode_name]
        trainers = sorted(mode_df["trainer_id"].unique())
        x = np.arange(len(trainers))
        width = 0.35

        for i, model_tag in enumerate(model_tags):
            sub = mode_df[mode_df["model"] == model_tag]
            vals = [
                sub[sub["trainer_id"] == t]["score_best"].values[0]
                if len(sub[sub["trainer_id"] == t]) > 0 else 0
                for t in trainers
            ]
            ax.bar(x + i * width, vals, width, label=model_tag)

        ax.set_title(f"Mode: {mode_name}")
        ax.set_xlabel("Trainer")
        ax.set_ylabel("Score (best, higher = better)")
        ax.set_xticks(x + width / 2)
        short = [t.replace("Algorithm", "") for t in trainers]
        ax.set_xticklabels(short, fontsize=9)
        ax.legend()
        ax.grid(True, alpha=0.3, axis="y")

    fig.suptitle("GSM8K — Model Comparison (score_best)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # Token efficiency: compare tokens per model if TB data available
    if has_tb:
        token_rows = []
        for m in all_tb_metrics:
            if m["tokens_in_final"] is None:
                continue
            token_rows.append({
                "model": m["model"],
                "trainer": m["trainer_id"].replace("Algorithm", ""),
                "mode": m["objective_mode"],
                "tokens_in": m["tokens_in_final"],
                "tokens_out": m["tokens_out_final"] or 0,
                "error": m["error_final"] or 1.0,
            })
        if token_rows:
            df_tok = pd.DataFrame(token_rows)
            df_tok["total_tokens"] = df_tok["tokens_in"] + df_tok["tokens_out"]

            print("\nToken efficiency by model:")
            tok_summary = df_tok.groupby("model").agg(
                mean_tokens_in=("tokens_in", "mean"),
                mean_tokens_out=("tokens_out", "mean"),
                mean_total=("total_tokens", "mean"),
                mean_error=("error", "mean"),
            ).round(1)
            print(tok_summary)

    # Numeric pivot
    print("\nScore comparison (score_best):")
    pivot = df_all.pivot_table(
        index=["trainer_id", "objective_mode"],
        columns="model",
        values="score_best",
        aggfunc="first",
    )
    pivot

## 10. Summary

In [ ]:
if df_all.empty:
    print("No results to summarize.")
else:
    print("=== GSM8K Multi-Objective Benchmark Summary ===")
    print(f"Total jobs: {len(df_all)}")
    print(f"Status: {df_all['status'].value_counts().to_dict()}")
    print(f"Models: {sorted(df_all['model'].unique())}")
    print(f"Modes: {sorted(df_all['objective_mode'].unique())}")
    print(f"Trainers: {sorted(df_all['trainer_id'].unique())}")

    ok_count = int((df_all["status"] == "ok").sum())
    total = len(df_all)
    print(f"\nSuccess rate: {ok_count}/{total} = {ok_count / total * 100:.0f}%")

    for model_tag in sorted(df_all["model"].unique()):
        for mode_name in ["weighted", "pareto"]:
            sub = df_all[(df_all["model"] == model_tag) & (df_all["objective_mode"] == mode_name)]
            ok_sub = sub[sub["status"] == "ok"]
            if ok_sub.empty:
                print(f"\n{model_tag} / {mode_name}: no OK jobs")
                continue
            best = ok_sub.loc[ok_sub["score_best"].idxmax()]
            print(f"\n{model_tag} / {mode_name}:")
            print(f"  Best trainer: {best['trainer_id']}")
            print(f"  score_best = {best['score_best']:.4f}")
            print(f"  time = {best['time_seconds']:.1f}s")

---

**Notes**:
- The GSM8K task uses a **BasicLearner** agent with a trainable system prompt.
  The LLM generates math solutions; the optimizer improves the system prompt.
- **UsageTrackingLLM** wraps the base LLM to intercept token counts from the
  provider's `response.usage` metadata. Falls back to whitespace-split estimates
  when usage metadata is missing.
- **TokenUsageAugmentingGuide** merges `tokens_in` / `tokens_out` into the
  score dict returned by `GSM8KGuide` (which provides `error`).
- Token weights (`1e-3`) are small relative to `error` weight (`1.0`) in weighted
  mode, reflecting that accuracy is the primary objective.
- Data is loaded from HuggingFace `openai/gsm8k` if available, otherwise from
  10 embedded examples.
- **API keys**: The setup cell auto-detects the provider from environment variables
  (loaded from `.env` via `python-dotenv` if available):
  - `OPENROUTER_API_KEY` — routes through OpenRouter (Xavier's default).
  - `XAI_API_KEY` + `DEEPSEEK_API_KEY` — direct provider access via LiteLLM.
  - No keys — falls back to stub mode with DummyLLM.
- See also: `06_multiobjective_convex.ipynb` (no LLM) and
  `07_multiobjective_bbeh.ipynb` (PAL code optimization with 2 objectives).